In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, group
from pyspark.sql import functions as F

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
cat, src = cfg["catalog"], group.source(cfg, env)
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print(("PASS " if ok else "FAIL ") + name + (f" -> {detail}" if detail and not ok else ""))

g = lambda t: spark.table(f"{cat}.gold.{t}")
f = lambda t: spark.table(f"{src}.gold.{t}")
print(cat, "group exposure + KPIs, FQF from", src)

In [0]:
ex, kpi, dim = g("group_exposure"), g("fact_group_kpi"), g("dim_group_customer")

n = ex.count()
check("one row per person", ex.select("group_customer_key").distinct().count() == n, n)
check("every person is in dim_group_customer", ex.join(dim, "group_customer_key", "left_anti").count() == 0)
check("total = FQF outstanding + BHG unpaid", ex.filter("total_exposure <> fqf_outstanding + bhg_unpaid").count() == 0)
check("no negative amounts, nobody listed with zero",
      ex.filter("fqf_outstanding < 0 OR bhg_unpaid < 0 OR total_exposure <= 0").count() == 0)
check("over_limit = above a known limit",
      ex.filter("over_limit <> (policy_limit IS NOT NULL AND total_exposure > policy_limit)").count() == 0)

fe = f("fact_fqf_event")
repaid = fe.filter("event_type = 'repayment'").groupBy("loan_key").agg(F.sum("principal_part").alias("r"))
expect = (f("dim_loan").filter("loan_key <> -1 AND status = 'active'").join(repaid, "loan_key", "left")
            .agg(F.sum(F.greatest(F.col("principal") - F.coalesce("r", F.lit(0)), F.lit(0)))).first()[0] or 0)
got = ex.agg(F.sum("fqf_outstanding")).first()[0] or 0
check("FQF outstanding = active loans minus principal repaid (FQF Gold)", abs(got - expect) < 0.01, f"{got} vs {expect}")
check("open loans = active loans in FQF Gold",
      ex.agg(F.sum("open_loans")).first()[0] == f("dim_loan").filter("loan_key <> -1 AND status = 'active'").count())

k = {r[0]: r[1] for r in kpi.select("kpi", "value").collect()}
check("10 KPIs, each once", len(k) == 10 and kpi.count() == 10, sorted(k))
check("KPI people = persons in dim_group_customer", k.get("people") == dim.filter("group_customer_key <> -1").count())
check("KPI people_over_limit = exposure table", k.get("people_over_limit") == ex.filter("over_limit").count())
check("KPI group_exposure = exposure table", abs(k.get("group_exposure", 0) - (ex.agg(F.sum("total_exposure")).first()[0] or 0)) < 0.01)

kpi.orderBy("kpi").show(truncate=False)
print("INFO people over their limit:")
ex.filter("over_limit").select("open_loans", "fqf_outstanding", "bhg_unpaid", "total_exposure", "policy_limit", "is_late").show()

print(f"\nt24_group_exposure: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t24_group_exposure failed")